# `consumer_complaints_1m`: workbench

The definition lives in [`dataset.py`](dataset.py) and the evidence in [`README.md`](README.md). Edit `dataset.py`
and re-run a cell: `ds` picks up the change (the raw files are not read again unless `_load_raw` changed).

In [ ]:
from data_foundry.v2 import workbench

ds = workbench()
ds.df.head()

In [ ]:
ds.run_all_checks()["summary"]

In [ ]:
ds.preview_splits()

## Exploration

Migrated from the v1 notebook; adapt the cells where needed.

In [ ]:
df = ds.df.copy()  # the migrated cells below work on `df`

In [ ]:
# Filter to year 2025
df_2025 = df[df[ds.task_metadata.time_on].dt.year == 2025].copy()

# Create a year-month column for grouping
df_2025["year_month"] = df_2025[ds.task_metadata.time_on].dt.to_period("M")

# 1) Total number of samples per month
monthly_totals = (
    df_2025
    .groupby("year_month")
    .size()
    .rename("total_samples")
)

# 2) Count of each class per month
monthly_class_counts = (
    df_2025
    .groupby(["year_month", ds.task_metadata.target_column_name])
    .size()
    .unstack(fill_value=0)
)

# Optional: combine both into a single DataFrame
result = monthly_class_counts.join(monthly_totals)

# Optional: sort by month and convert PeriodIndex to timestamp (month start)
result = result.sort_index()
result.index = result.index.to_timestamp()

In [ ]:
result

## Report

`ds.check()` runs the full pipeline and the bundle checks without saving and writes `README.md`. `ds.build()` also
saves the container (new UUID); only the curator runs it.

In [ ]:
result = ds.check()